# exp<NN>r<R> — &lt;short title&gt;

**COPY THIS FOLDER to start an experiment:**
`cp -r experiments/_template experiments/exp01r1`

Then fill in the four fields below and delete this instruction block.

---

**Round** &lt;R&gt; (&lt;scale&gt; data, &lt;N&gt; shards, &lt;budget&gt;s) · **Parent** &lt;exp id this is a delta against&gt;

**Hypothesis** — one sentence, falsifiable. State what should move and in which
direction. "Try a lower LR" is not a hypothesis; "warmdown 0.5 is tuned for ~950 steps,
so at ~380 steps a shorter warmdown should win" is.

**Change** — the delta against the parent, exactly. One axis at a time. If you changed
two things, that is two experiments.

## 1 · Setup

`prepare_round()` runs first, before any dataloader or eval. The harness asserts if you
forget, because the alternative is a run at the wrong data scale that still produces a
plausible-looking `val_bpb`.

In [ ]:
import sys
sys.path.insert(0, "/home/dangnguyenducanh/autoresearch")

import prepare
from experiments._shared.data import prepare_round, ROUND_BUDGET
from experiments._shared.harness import ExpConfig, run, run_seeds
from experiments._shared.ledger import verdict, retest_verdict, append_result

EXP_ID = "exp01r1"          # == this folder's name
ROUND  = 1
OUTDIR = f"experiments/{EXP_ID}"

# From rounds/r<R>-directions.md — measured on THIS round's baseline, not inherited.
BASELINE_BPB = 1.012000     # <-- fill from the round baseline
SIGMA        = 0.0055       # <-- fill: sigma for this round

prepare_round(ROUND)

## 2 · Config

Start from the parent's config and change one thing. The printed dict is what the report
cites — if a number is not in here, the run is not reproducible.

In [ ]:
import dataclasses

parent = ExpConfig(
    round_idx=ROUND,
    time_budget=ROUND_BUDGET[ROUND],
    gpu="H100",
    # ... the parent experiment's winning values go here ...
)

cfg = dataclasses.replace(
    parent,
    matrix_lr=0.02,                       # <-- THE CHANGE. one axis.
    notes="matrix_lr 0.04 -> 0.02",
)

for k, v in vars(cfg).items():
    marker = "  <-- changed" if getattr(parent, k) != v else ""
    print(f"  {k:20s} {v}{marker}")

## 3 · Train

Single seed first. Only spend the extra seeds if the result lands near the noise band —
that is what `verdict` tells you.

In [ ]:
m = run(cfg, outdir=OUTDIR)

## 4 · Verdict

You do not decide this by eye. `verdict()` is §4 of `research-program.md` as code.

Set `simpler=True` only if the change made the code strictly smaller or removed a
concept — that is the one case where a result inside the noise band still advances.

In [ ]:
v = verdict(
    val_bpb=m["val_bpb"],
    baseline_bpb=BASELINE_BPB,
    sigma=SIGMA,
    simpler=False,        # True only if this change DELETED code / removed a concept
)
print(v)

if v.status == "retest":
    print("\nNear the band — running 2 more seeds before deciding.")
    mean, sd, runs = run_seeds(cfg, seeds=(1, 2), outdir=OUTDIR)
    all_bpbs = [m["val_bpb"]] + [r["val_bpb"] for r in runs]
    v = retest_verdict(sum(all_bpbs) / len(all_bpbs), BASELINE_BPB, SIGMA)
    print(v)

## 5 · Record

The ledger row and the report are both required. A run without `report.md` is not done.

In [ ]:
append_result(
    exp_id=EXP_ID, round_idx=ROUND, metrics=m, verdict=v,
    description=cfg.notes,        # no tabs; commas are fine
)
print(open("results.tsv").read())

## 6 · Then, outside the notebook

1. Write `report.md` — copy the structure in the template file, fill every section.
   Numbers come from `metrics.json`, never retyped from memory.
2. `keep` → `git commit`, the branch advances, this becomes the round baseline.
   `discard` / `crash` → `git reset --hard` back to the round baseline.
3. `epoch` in the summary must be < 1.0. If it is not, the run is invalid regardless of
   its `val_bpb` — raise `SHARD_BUDGET` and rerun.